# Module 2: Classical AI - Search Algorithms

In classical Artificial Intelligence, problem-solving is often framed as a state-space search. A search algorithm explores a graph of states (starting from an initial state) to find a path to a goal state.

In this notebook, we will implement and compare three fundamental search algorithms:
1. **Breadth-First Search (BFS)**: Explores level by level (guarantees shortest path in unweighted graphs).
2. **Depth-First Search (DFS)**: Explores as deep as possible before backtracking.
3. **A* Search**: An informed (heuristic) search that uses both path cost and estimated distance to the goal to find the shortest path efficiently.

We will apply these algorithms to solve a **grid-based maze pathfinding problem**.

In [ ]:
from collections import deque
import heapq
import numpy as np
import matplotlib.pyplot as plt

---

## Define the Maze Environment

We will represent a maze as a 2D numpy array where:
- `0` represents a free navigable space.
- `1` represents an obstacle / wall.

We will also define a start coordinate and a goal coordinate.

In [ ]:
# 0 = path, 1 = wall
maze = np.array([
    [0, 0, 1, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 1, 0, 1, 1, 1, 1, 0, 0],
    [0, 0, 0, 0, 1, 0, 0, 1, 0, 0],
    [0, 1, 1, 0, 1, 0, 0, 1, 1, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
    [0, 1, 1, 1, 1, 1, 1, 0, 1, 0],
    [0, 0, 0, 0, 0, 0, 1, 0, 1, 0],
    [0, 1, 1, 1, 1, 0, 1, 0, 0, 0],
    [0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 1, 1, 1, 0]
])

start = (0, 0)
goal = (9, 9)

# Function to display the maze
def plot_maze(maze, path=None, title="Maze"):
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(maze, cmap='binary')
    
    # Draw start and goal
    ax.plot(start[1], start[0], 'go', markersize=12, label='Start')
    ax.plot(goal[1], goal[0], 'ro', markersize=12, label='Goal')
    
    if path:
        py, px = zip(*path)
        ax.plot(px, py, 'b-', linewidth=3, label='Path')
        
    ax.set_xticks(range(maze.shape[1]))
    ax.set_yticks(range(maze.shape[0]))
    ax.grid(color='gray', linestyle='-', linewidth=0.5)
    ax.legend()
    plt.title(title)
    plt.show()

plot_maze(maze, title="Initial Maze Environment")

---

## 1. Breadth-First Search (BFS)

BFS uses a **Queue (FIFO)** data structure to store frontier nodes. It explores all neighbors of the current depth before moving to the next level.

**Complexity:** $O(V + E)$ where $V$ is vertices (states) and $E$ is edges (valid moves).

In [ ]:
def bfs(maze, start, goal):
    rows, cols = maze.shape
    queue = deque([start])
    came_from = {start: None}
    
    # Valid movement directions (Up, Down, Left, Right)
    directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    
    while queue:
        current = queue.popleft()
        
        if current == goal:
            break
            
        for dr, dc in directions:
            neighbor = (current[0] + dr, current[1] + dc)
            
            # Check grid boundaries and obstacle collision
            if (0 <= neighbor[0] < rows) and (0 <= neighbor[1] < cols):
                if maze[neighbor] == 0 and neighbor not in came_from:
                    queue.append(neighbor)
                    came_from[neighbor] = current
                    
    # Reconstruct path
    if goal not in came_from:
        return None # No path found
        
    path = []
    curr = goal
    while curr is not None:
        path.append(curr)
        curr = came_from[curr]
    return path[::-1]

bfs_path = bfs(maze, start, goal)
plot_maze(maze, bfs_path, "BFS Pathfinding Solution")

---

## 2. Depth-First Search (DFS)

DFS uses a **Stack (LIFO)** data structure. It explores deep along each branch before backtracking.

*Note: In grid-world mazes, DFS often returns suboptimal paths (longer paths).* 

In [ ]:
def dfs(maze, start, goal):
    rows, cols = maze.shape
    stack = [start]
    came_from = {start: None}
    directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    
    while stack:
        current = stack.pop()
        
        if current == goal:
            break
            
        for dr, dc in directions:
            neighbor = (current[0] + dr, current[1] + dc)
            
            if (0 <= neighbor[0] < rows) and (0 <= neighbor[1] < cols):
                if maze[neighbor] == 0 and neighbor not in came_from:
                    stack.append(neighbor)
                    came_from[neighbor] = current
                    
    if goal not in came_from:
        return None
        
    path = []
    curr = goal
    while curr is not None:
        path.append(curr)
        curr = came_from[curr]
    return path[::-1]

dfs_path = dfs(maze, start, goal)
plot_maze(maze, dfs_path, "DFS Pathfinding Solution")

---

## 3. A* Search (Informed Search)

A* selects the node that minimizes a total evaluation cost function $f(n)$:
$$f(n) = g(n) + h(n)$$
where:
- $g(n)$ is the cost to reach node $n$ from the start.
- $h(n)$ is the heuristic function estimating the cost from $n$ to the goal (e.g., Manhattan distance).

To solve this, A* uses a **Priority Queue (Min-Heap)**.

In [ ]:
def manhattan_distance(p1, p2):
    return abs(p1[0] - p2[0]) + abs(p1[1] - p2[1])

def astar(maze, start, goal):
    rows, cols = maze.shape
    
    # Priority queue holds entries of: (f_score, state_coord)
    frontier = []
    heapq.heappush(frontier, (0, start))
    
    came_from = {start: None}
    g_score = {start: 0}
    
    directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    
    while frontier:
        # Get the node with lowest f(n) = g(n) + h(n)
        _, current = heapq.heappop(frontier)
        
        if current == goal:
            break
            
        for dr, dc in directions:
            neighbor = (current[0] + dr, current[1] + dc)
            
            if (0 <= neighbor[0] < rows) and (0 <= neighbor[1] < cols):
                if maze[neighbor] == 1:
                    continue
                    
                # Step cost from current to neighbor is 1
                tentative_g_score = g_score[current] + 1
                
                if neighbor not in g_score or tentative_g_score < g_score[neighbor]:
                    g_score[neighbor] = tentative_g_score
                    f_score = tentative_g_score + manhattan_distance(neighbor, goal)
                    heapq.heappush(frontier, (f_score, neighbor))
                    came_from[neighbor] = current

    if goal not in came_from:
        return None
        
    path = []
    curr = goal
    while curr is not None:
        path.append(curr)
        curr = came_from[curr]
    return path[::-1]

astar_path = astar(maze, start, goal)
plot_maze(maze, astar_path, "A* Search Solution")

---

## Comparison
Let's compare the lengths of the paths found by each algorithm.

In [ ]:
print(f"BFS path length: {len(bfs_path) if bfs_path else 'No Path'}")
print(f"DFS path length: {len(dfs_path) if dfs_path else 'No Path'}")
print(f"A* path length: {len(astar_path) if astar_path else 'No Path'}")

### 🏋️ Exercise: Euclidean Distance Heuristic

In A* Search, we used the Manhattan distance heuristic:
$$h_{\text{Manhattan}}(n) = |x_n - x_{\text{goal}}| + |y_n - y_{\text{goal}}|$$

Implement the **Euclidean distance** heuristic instead:
$$h_{\text{Euclidean}}(n) = \sqrt{(x_n - x_{\text{goal}})^2 + (y_n - y_{\text{goal}})^2}$$

Write a modified `astar_euclidean` function and verify if it finds a path of the same length.

In [ ]:
import math

def euclidean_distance(p1, p2):
    # YOUR CODE HERE
    pass

# Test your heuristic with a custom A* function or compare the output here